# 04 · Post-hoc analyses and figures (CPU, no training)
Duplicate leakage with Fitzpatrick17k-C, leak-free interventions, DDI at matched specificity, pooled-over-seeds statistics (Wilson and bootstrap intervals, contrasts, matched-specificity threshold, trend test, site-adjusted regression, subtype intervals, paired intervention differences) and the paper figures.

**Kaggle inputs:** notebook 02 and notebook 03 saved outputs; `fitzpatrick17k` (Nazmus Resan) for the label CSV. Internet on (downloads the Fitzpatrick17k-C cluster lists from GitHub).
**Runtime:** about 20–30 min. **Outputs:** `figs_final/`, `posthoc2.log`.

## 1. Gather inputs and download the Fitzpatrick17k-C lists

In [ ]:
%%writefile setup_inputs.py
"""Gather everything the post-hoc analyses need into the working folder (CPU, ~1 min, internet on)."""
import glob, json, os, shutil, urllib.request
IN = os.environ.get("INPUT_ROOT", "/kaggle/input")
W = os.environ.get("WORK", "/kaggle/working")
os.makedirs(f"{W}/zipin", exist_ok=True)
RAW = ("https://raw.githubusercontent.com/kakumarabhishek/Corrected-Skin-Image-Datasets/main/"
       "Fitzpatrick17k/Fitzpatrick17k_Analysis/")
files = {"complete_clusters.txt": "AnalysesOutputFiles/complete_clusters.txt",
         "f17k_names.csv": "DatasetSplits/SimThresh_F_A2_F_0.99_0.70_FC_F_KeepOne_Out_F_OutThresh_None_0FST_F.csv",
         "f17k_C.csv": "DatasetSplits/SimThresh_T_A2_T_0.99_0.70_FC_T_KeepOne_Out_T_OutThresh_None_0FST_F.csv"}
for name, path in files.items():
    if not os.path.exists(f"{W}/{name}"):
        urllib.request.urlretrieve(RAW + path, f"{W}/{name}")
    print(f"[setup] {name}: {os.path.getsize(f'{W}/{name}'):,} bytes")
csv = [c for c in glob.glob(f"{IN}/**/*.csv", recursive=True) if os.path.basename(c) == "fitzpatrick17k (1).csv"]
if not csv:
    raise FileNotFoundError("Attach the Nazmus Resan fitzpatrick17k dataset (for its CSV)")
shutil.copy(csv[0], f"{W}/official.csv")
n = skipped = 0
for jf in glob.glob(f"{IN}/**/results_seed*.json", recursive=True):
    if "smoke" in jf:
        continue
    if "original" not in str(json.load(open(jf)).get("config", {}).get("img_root", "")).lower():
        skipped += 1; continue
    pf = jf.replace("results_", "probs_").replace(".json", ".csv")
    if os.path.exists(pf) and not os.path.exists(f"{W}/zipin/{os.path.basename(pf)}"):
        shutil.copy(pf, f"{W}/zipin/"); n += 1
print(f"[setup] copied {n} original-image probability files (skipped {skipped} masked-image runs)")
print(sorted(os.listdir(f"{W}/zipin")))
ddi = glob.glob(f"{IN}/**/probs_ddi_all.csv", recursive=True)
if ddi:
    shutil.copy(ddi[0], f"{W}/probs_ddi_all.csv"); print("[setup] probs_ddi_all.csv copied")
else:
    print("[setup] probs_ddi_all.csv not found: attach notebook 03 (DDI) to run the DDI analysis")

In [ ]:
%run setup_inputs.py

## 2. Write the scripts

In [ ]:
%%writefile analyze_dups.py
"""
Duplicate-leakage check using Fitzpatrick17k-C (Abhishek et al. 2025).
For each seed: a test image is 'leaked' if a duplicate of it (same cluster) was in that seed's training set.
Recompute per-FST results (a) excluding leaked test images, (b) on Fitzpatrick17k-C images only.
Usage: python analyze_dups.py <dir with probs_seed{N}{suffix}.csv> [suffix]
"""
import ast, glob, os, re, sys
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

H = os.path.dirname(os.path.abspath(__file__))
P, SUF = sys.argv[1], (sys.argv[2] if len(sys.argv) > 2 else "")
NB = int(os.environ.get("NBOOT", 1000))
rng = np.random.default_rng(0)
pd.set_option("display.width", 200, "display.precision", 3)

off = pd.read_csv(f"{H}/official.csv"); off = off[off.fitzpatrick_scale.between(1, 6)]
fst = dict(zip(off.md5hash, off.fitzpatrick_scale.astype(int)))
names = pd.read_csv(f"{H}/f17k_names.csv"); n2m = dict(zip(names.filename, names.md5hash))
cl = [[n2m[n] for n in ast.literal_eval(l) if n in n2m] for l in open(f"{H}/complete_clusters.txt") if l.strip()]
cid = {h: i for i, c in enumerate(cl) for h in c}
members = {i: set(c) for i, c in enumerate(cl)}
keepC = set(pd.read_csv(f"{H}/f17k_C.csv").md5hash)

runs = {}
for f in glob.glob(f"{P}/probs_seed*{SUF}.csv"):
    m = re.fullmatch(rf"probs_seed(\d+){re.escape(SUF)}\.csv", os.path.basename(f))
    if m: runs[int(m.group(1))] = pd.read_csv(f)
if not runs: sys.exit(f"no probs_seed*{SUF}.csv in {P}")
print(f"seeds {sorted(runs)}  suffix='{SUF}'")

light_all = {h for h, f in fst.items() if f <= 3}
for s, d in runs.items():
    d["md5hash"] = d.path.map(lambda p: os.path.splitext(os.path.basename(str(p)))[0])
    train = light_all - set(d.loc[d.fst <= 3, "md5hash"])          # baseline: train = light images not in test
    if "hold45" in SUF:                                              # holdout runs also trained on every FST IV-V image
        train |= {h for h, f in fst.items() if f in (4, 5)}         # (size-matched: superset of its light images -> conservative)
    def leaked(h):
        i = cid.get(h)
        return i is not None and bool((members[i] - {h}) & train)
    d["leaked"] = d.md5hash.map(leaked)
    d["inC"] = d.md5hash.isin(keepC)


def grp(f): return np.where(f <= 3, "I-III", np.where(f <= 5, "IV-V", "VI"))
def sens(p, y): return float((p[y == 1] >= .5).mean()) if (y == 1).any() else np.nan
def auc(p, y): return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan


print("\nleaked test images (a duplicate was in training), mean per seed")
L = pd.concat([d.assign(seed=s) for s, d in runs.items()], ignore_index=True)
L["g"] = grp(L.fst)
L["leaked_mal"] = L.leaked & (L.y == 1)
L["mal"] = L.y == 1
print(L.groupby("g")[["leaked", "mal", "leaked_mal"]].agg(["size", "sum"]).iloc[:, [0, 1, 3, 5]]
      .set_axis(["test", "leaked", "malignant", "leaked_malignant"], axis=1).div(len(runs)).round(1).to_string())

for name, keep in (("ALL TEST IMAGES", lambda d: d), ("EXCLUDING LEAKED TEST IMAGES", lambda d: d[~d.leaked]),
                   ("FITZPATRICK17K-C IMAGES ONLY", lambda d: d[d.inC]),
                   ("FITZPATRICK17K-C AND NOT LEAKED", lambda d: d[d.inC & ~d.leaked])):
    rows = []
    for s, d in runs.items():
        x = keep(d)
        for f, z in x.groupby("fst"):
            rows.append(dict(seed=s, fst=int(f), n=len(z), pos=int(z.y.sum()),
                             sens=sens(z.prob.values, z.y.values), auc=auc(z.prob.values, z.y.values)))
    T = pd.DataFrame(rows)
    print("\n" + "=" * 70 + f"\n{name}\n" + "=" * 70)
    print(T.groupby("fst").agg(n=("n", "mean"), pos=("pos", "mean"), sens=("sens", "mean"),
                               sens_sd=("sens", "std"), auc=("auc", "mean"), auc_sd=("auc", "std")).round(3).to_string())
    pairs = (("I-III", "VI"),) if "hold45" in SUF else (("I-III", "VI"), ("I-III", "IV-V"), ("IV-V", "VI"))
    for lo, hi in pairs:
        for metric, st in (("sensitivity", sens), ("AUC", auc)):
            res = []
            for s, d in runs.items():
                x = keep(d); g = grp(x.fst)
                A, B = x[g == lo], x[g == hi]
                pt = st(A.prob.values, A.y.values) - st(B.prob.values, B.y.values)
                bs = []
                for _ in range(NB):
                    i = rng.integers(0, len(A), len(A)); j = rng.integers(0, len(B), len(B))
                    bs.append(st(A.prob.values[i], A.y.values[i]) - st(B.prob.values[j], B.y.values[j]))
                lo_, hi_ = np.nanpercentile(bs, [2.5, 97.5]); res.append((pt, lo_ > 0 or hi_ < 0))
            print(f"  {lo:>5s} minus {hi:<4s} {metric:11s} mean gap {np.mean([r[0] for r in res]):+.3f}  "
                  f"CI excludes 0 in {sum(r[1] for r in res)}/{len(res)} seeds")

In [ ]:
%%writefile intervention_leakfree.py
"""Table III: baseline vs +FST IV-V vs size-matched on the same leak-free FST VI images.
Also writes leak-free probability files to lf/out for the Fig. 3 script."""
import ast, os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
n2m = dict(zip(*[pd.read_csv("f17k_names.csv")[c] for c in ("filename", "md5hash")]))
o = pd.read_csv("official.csv"); o = o[o.fitzpatrick_scale.between(1, 6)]
fst = dict(zip(o.md5hash, o.fitzpatrick_scale))
cl = [[n2m[n] for n in ast.literal_eval(l) if n in n2m] for l in open("complete_clusters.txt") if l.strip()]
cid = {h: i for i, c in enumerate(cl) for h in c}; mem = {i: set(c) for i, c in enumerate(cl)}
keepC = set(pd.read_csv("f17k_C.csv").md5hash)
light = {h for h, f in fst.items() if f <= 3}; ivv = {h for h, f in fst.items() if f in (4, 5)}
os.makedirs("lf/out", exist_ok=True)
stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
rows = []
for s in range(3):
    P = {k: pd.read_csv(f"zipin/probs_seed{s}{k}.csv") for k in ("", "_hold45", "_hold45_sm")}
    for d in P.values(): d["h"] = d.path.map(stem)
    train = (light - set(P[""].loc[P[""].fst <= 3, "h"])) | ivv
    bad = {h for h in P[""].loc[P[""].fst == 6, "h"] if (i := cid.get(h)) is not None and (mem[i] - {h}) & train}
    for k, d in P.items():
        d[~((d.fst == 6) & d.h.isin(bad))].drop(columns="h").to_csv(f"lf/out/probs_seed{s}{k}.csv", index=False)
    six = P[""][P[""].fst == 6]
    for sub, H in (("all FST VI", set(six.h)), ("leak-free FST VI", set(six.h) - bad),
                   ("F17k-C & leak-free", (set(six.h) - bad) & keepC)):
        for k, lab in (("", "baseline"), ("_hold45", "+IV-V"), ("_hold45_sm", "+IV-V same size")):
            z = P[k][P[k].h.isin(H) & (P[k].fst == 6)]
            L = P[k][P[k].fst <= 3]
            rows.append(dict(seed=s, subset=sub, model=lab, n=len(z), pos=int(z.y.sum()),
                             sens=(z.prob[z.y == 1] >= .5).mean(), auc=roc_auc_score(z.y, z.prob),
                             light_sens=(L.prob[L.y == 1] >= .5).mean(), light_spec=(L.prob[L.y == 0] < .5).mean(),
                             light_auc=roc_auc_score(L.y, L.prob)))
    print(f"seed {s}: {len(bad)} FST VI test images have a near-duplicate in the holdout training set")
T = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(T.groupby(["subset", "model"], sort=False)[["n", "pos", "sens", "auc", "light_sens", "light_spec", "light_auc"]]
      .mean().round(3).to_string())
for sub in T.subset.unique():
    x = T[T.subset == sub]
    for m in ("+IV-V", "+IV-V same size"):
        g = (x[x.model == m].set_index("seed")[["sens", "auc"]] - x[x.model == "baseline"].set_index("seed")[["sens", "auc"]])
        print(f"{sub:20s} {m:16s} per-seed gain  sens {g.sens.round(3).tolist()}  auc {g.auc.round(3).tolist()}")

In [ ]:
%%writefile ddi_matched.py
import sys
import numpy as np, pandas as pd
P = pd.read_csv(sys.argv[1])
rng = np.random.default_rng(0)
G = ["I-II", "III-IV", "V-VI"]
def sens_at_spec(p, y, spec):
    neg = np.sort(p[y == 0]); pos = p[y == 1]
    thr = np.quantile(neg, spec)           # spec fraction of negatives fall <= thr
    return float((pos > thr).mean())
def boot_gap(L, D, spec, nb=500):
    f = lambda d: sens_at_spec(d.prob.values, d.y.values, spec)
    pt = f(L) - f(D); g = []
    for _ in range(nb):
        a = L.iloc[rng.integers(0, len(L), len(L))]; b = D.iloc[rng.integers(0, len(D), len(D))]
        g.append(f(a) - f(b))
    return pt, np.percentile(g, 2.5), np.percentile(g, 97.5)
for spec in (0.8, 0.9):
    print(f"\n=== sensitivity at {int(spec*100)}% specificity, set WITHIN each group (mean over seeds) ===")
    rows = []
    for (cfg, s, g), d in P.groupby(["cfg", "seed", "grp"]):
        rows.append(dict(cfg=cfg, seed=s, grp=g, sens=sens_at_spec(d.prob.values, d.y.values, spec)))
    T = pd.DataFrame(rows)
    print(T.pivot_table(index="cfg", columns="grp", values="sens").reindex(columns=G).round(3).to_string())
    print(f"--- contrasts at {int(spec*100)}% spec (bootstrap per seed) ---")
    for cfg, d in P.groupby("cfg"):
        for lo, hi in (("I-II", "V-VI"), ("III-IV", "V-VI")):
            res = [boot_gap(x[x.grp == lo], x[x.grp == hi], spec) for _, x in d.groupby("seed")]
            sig = sum(1 for r in res if r[1] > 0 or r[2] < 0)
            print(f"{cfg:20s} {lo:>6s} minus {hi:<6s} mean gap {np.mean([r[0] for r in res]):+.3f}  CI excludes 0 in {sig}/{len(res)}")
# ensemble over seeds (mean prob per image) for the two main configs
print("\n=== seed-ensemble AUC (mean prob per image) ===")
from sklearn.metrics import roc_auc_score
for cfg, d in P.groupby("cfg"):
    E = d.groupby(["file", "grp", "y"], as_index=False).prob.mean()
    print(cfg, {g: round(roc_auc_score(x.y, x.prob), 3) for g, x in E.groupby("grp")})

In [ ]:
%%writefile new_figs.py
"""Extra paper figures: study design, grouping comparison, robustness forest plot."""
import ast, os, glob, re
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import roc_auc_score

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
                     "font.size": 7.5, "axes.labelsize": 7.5, "axes.titlesize": 8, "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "legend.fontsize": 6.5, "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 300})
BASE, HI, REF = "#7A9CC6", "#D55E00", "#333333"
OUT = "figs_extra"; os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(0)
NB = int(os.environ.get("NBOOT", 1000))


def save(fig, name):
    fig.savefig(f"{OUT}/{name}.pdf", bbox_inches="tight"); fig.savefig(f"{OUT}/{name}.png", bbox_inches="tight")
    plt.close(fig); print("wrote", name)


# ---------------------------------------------------------------- data
off = pd.read_csv("official.csv"); off = off[off.fitzpatrick_scale.between(1, 6)]
cent = dict(zip(off.md5hash, off.fitzpatrick_centaur))
n2m = dict(zip(*[pd.read_csv("f17k_names.csv")[c] for c in ("filename", "md5hash")]))
cl = [[n2m[n] for n in ast.literal_eval(l) if n in n2m] for l in open("complete_clusters.txt") if l.strip()]
cid = {h: i for i, c in enumerate(cl) for h in c}; mem = {i: set(c) for i, c in enumerate(cl)}
keepC = set(pd.read_csv("f17k_C.csv").md5hash)
light_all = set(off.md5hash[off.fitzpatrick_scale <= 3])


def load(suf):
    R = {}
    for f in glob.glob(f"zipin/probs_seed*{suf}.csv"):
        m = re.fullmatch(rf"probs_seed(\d+){re.escape(suf)}\.csv", os.path.basename(f))
        if not m: continue
        d = pd.read_csv(f); d["h"] = d.path.map(lambda p: os.path.splitext(os.path.basename(p))[0])
        train = light_all - set(d.h[d.fst <= 3])
        d["leaked"] = d.h.map(lambda h: (i := cid.get(h)) is not None and bool((mem[i] - {h}) & train))
        d["inC"] = d.h.isin(keepC); d["fc"] = d.h.map(cent)
        R[int(m.group(1))] = d
    return R


main, eff = load(""), load("_effb0")
sens = lambda p, y: float((p[y == 1] >= .5).mean())
auc = lambda p, y: float(roc_auc_score(y, p))

# ---------------------------------------------------------------- Fig: grouping comparison
groups = [("Light / dark", [("I–III", [1, 2, 3]), ("IV–VI", [4, 5, 6])]),
          ("Pairs (as DDI)", [("I–II", [1, 2]), ("III–IV", [3, 4]), ("V–VI", [5, 6])]),
          ("Per type", [(r, [i]) for i, r in enumerate(["I", "II", "III", "IV", "V", "VI"], 1)])]
fig, ax = plt.subplots(figsize=(3.5, 1.95))
x, ticks, labs, centres = 0.0, [], [], []
for gname, bars in groups:
    start = x
    for lab, types in bars:
        v = np.array([sens(d.prob[d.fst.isin(types)].values, d.y[d.fst.isin(types)].values) for d in main.values()])
        isvi = types == [6]
        has6 = 6 in types
        ax.bar(x, v.mean(), 0.8, color=HI if isvi else BASE, hatch="////" if isvi else None,
               edgecolor="black", linewidth=0.5, alpha=1 if (isvi or not has6) else 0.55)
        ax.errorbar(x, v.mean(), v.std(ddof=1), color="black", capsize=1.5, lw=0.6)
        ax.text(x, v.mean() + v.std(ddof=1) + 0.02, f"{v.mean():.2f}", ha="center", fontsize=5.5)
        ticks.append(x); labs.append(lab); x += 1.3 if len(bars) < 6 else 1.0
    centres.append(((start + x - (1.3 if len(bars) < 6 else 1.0)) / 2, gname)); x += 0.9
ax.set_xticks(ticks); ax.set_xticklabels(labs, fontsize=5.6)
ax.set_ylabel("Sensitivity"); ax.set_ylim(0, 0.95)
for c, g in centres:
    ax.text(c, -0.19, g, ha="center", va="top", fontsize=6.2, style="italic", transform=ax.get_xaxis_transform())
save(fig, "fig_grouping")

# ---------------------------------------------------------------- Fig: robustness forest plot
def gap_ci(runs, filt, lo_types, hi_types, stat, fcol="fst"):
    pts, los, his = [], [], []
    for d in runs.values():
        x = filt(d); x = x[x[fcol].notna()]
        A, B = x[x[fcol].isin(lo_types)], x[x[fcol].isin(hi_types)]
        pa, ya, pb, yb = A.prob.values, A.y.values, B.prob.values, B.y.values
        pts.append(stat(pa, ya) - stat(pb, yb))
        bs = [stat(pa[i], ya[i]) - stat(pb[j], yb[j]) for i, j in
              ((rng.integers(0, len(ya), len(ya)), rng.integers(0, len(yb), len(yb))) for _ in range(NB))]
        l, h = np.percentile(bs, [2.5, 97.5]); los.append(l); his.append(h)
    return np.mean(pts), np.mean(los), np.mean(his)


checks = [("All test images", main, lambda d: d, "fst"),
          ("EfficientNet-B0", eff, lambda d: d, "fst"),
          ("Second annotator", main, lambda d: d, "fc"),
          ("Both annotators agree", main, lambda d: d[d.fst == d.fc], "fst"),
          ("Duplicates removed", main, lambda d: d[~d.leaked], "fst"),
          ("Fitzpatrick17k-C only", main, lambda d: d[d.inC], "fst")]
fig, axes = plt.subplots(1, 2, figsize=(3.5, 1.75), sharey=True)
for ax, (title, stat) in zip(axes, (("(a) Sensitivity gap", sens), ("(b) AUC gap", auc))):
    for k, (name, runs, filt, col) in enumerate(checks):
        y = len(checks) - 1 - k
        for off_, (lo_t, hi_t, mk, colr) in zip((0.15, -0.15), (([1, 2, 3], [6], "o", HI), ([4, 5], [6], "s", BASE))):
            p, l, h = gap_ci(runs, filt, lo_t, hi_t, stat, col)
            ax.errorbar(p, y + off_, xerr=[[p - l], [h - p]], fmt=mk, ms=3, color=colr, mec="black", mew=0.4,
                        ecolor="black", elinewidth=0.6, capsize=1.2)
    ax.axvline(0, color=REF, lw=0.6, ls="--")
    ax.set_title(title, loc="left", fontsize=7.5); ax.set_xlim(left=-0.05)
    ax.set_yticks(range(len(checks))); ax.set_yticklabels([c[0] for c in checks][::-1], fontsize=6.3)
from matplotlib.lines import Line2D
axes[0].legend(handles=[Line2D([], [], marker="o", ls="", color=HI, mec="black", label="FST I–III minus VI"),
                        Line2D([], [], marker="s", ls="", color=BASE, mec="black", label="FST IV–V minus VI")],
               loc="upper center", bbox_to_anchor=(1.0, -0.13), ncol=2, frameon=False, fontsize=6)
save(fig, "fig_forest")

# ---------------------------------------------------------------- Fig: study design
from matplotlib.patches import Rectangle
LB, LF, DB, DF = "#4C72B0", "#EEF3FA", "#D55E00", "#FCEEE6"        # light-path / dark-path edge + fill
TONES = ["#F3D9C4", "#E6BE9C", "#CC9C74", "#A06D48", "#704630", "#3E271C"]   # FST I..VI swatches
fig, ax = plt.subplots(figsize=(7.16, 2.15)); ax.set_xlim(0, 100); ax.set_ylim(0, 33); ax.axis("off")
def box(x, y, w, h, title, body="", ec=LB, fc=LF, bfs=6.8, tfs=7.6):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.2,rounding_size=0.9", fc=fc, ec=ec, lw=0.9))
    ax.text(x + w / 2, y + h - 1.4, title, ha="center", va="top", fontsize=tfs, weight="bold")
    if body:
        ax.text(x + w / 2, y + h - 4.8, body, ha="center", va="top", fontsize=bfs, linespacing=1.3)
def arrow(pts, color=REF, ls="-"):
    xs, ys = zip(*pts)
    if len(pts) > 2: ax.plot(xs[:-1], ys[:-1], color=color, lw=0.8, ls=ls)
    ax.annotate("", pts[-1], pts[-2], arrowprops=dict(arrowstyle="-|>", lw=0.8, color=color, ls=ls,
                                                       shrinkA=0, shrinkB=0, mutation_scale=7))
box(0.4, 9, 15.2, 15, "Fitzpatrick17k", "16,012 images\nFST I–VI\n2,160 malignant", ec=REF, fc="#F5F5F5", tfs=7.0)
box(18.8, 18.2, 15.8, 10.5, "FST I–III", "train 8,851\ntest 2,212")
box(18.8, 1.2, 15.8, 13.3, "FST IV–VI", "test only\nIV 2,781 · V 1,533\nVI 635", ec=DB, fc=DF, bfs=6.3)
box(37.5, 13, 15, 13, "Train", "ResNet-50\nEfficientNet-B0\n5 seeds")
box(55.2, 9, 19.5, 17, "Evaluate per type")
for i, c in enumerate(TONES):                                    # six skin-type swatches
    x0 = 56.8 + i * 2.85
    ax.add_patch(Rectangle((x0, 15.4), 2.45, 4.2, fc=c, ec=DB if i == 5 else "white", lw=1.4 if i == 5 else 0.4))
    ax.text(x0 + 1.22, 14.6, ["I", "II", "III", "IV", "V", "VI"][i], ha="center", va="top", fontsize=6.3,
            color=DB if i == 5 else "black", weight="bold" if i == 5 else "normal")
ax.text(64.95, 11.6, "sensitivity · AUC", ha="center", va="top", fontsize=6.8)
box(78.0, 22.4, 21.6, 9.6, "External check", "DDI, biopsy-confirmed\nno retraining")
box(78.0, 11.6, 21.6, 9.6, "Robustness", "subtype · source site\nannotator · duplicates")
box(78.0, 0.8, 21.6, 9.6, "Interventions", "+FST IV–V in training\nsize-matched · 3 seeds")
arrow([(15.9, 19), (18.5, 23.4)]); arrow([(15.9, 14), (18.5, 7.8)])
arrow([(34.9, 23.4), (37.2, 21.5)], color=LB)
arrow([(52.8, 19.5), (54.9, 19.5)])
arrow([(34.9, 5.0), (64.95, 5.0), (64.95, 8.7)], color=DB, ls="--")
ax.text(50.0, 4.3, "held out: never trained on", fontsize=6.2, style="italic", ha="center", va="top", color=DB)
arrow([(45.0, 26.3), (45.0, 31.2), (76.2, 31.2), (76.2, 27.2), (77.7, 27.2)])
arrow([(75.0, 17.5), (77.7, 16.4)]); arrow([(75.0, 14.0), (77.7, 5.6)])
save(fig, "fig_design")


In [ ]:
%%writefile make_figures.py
"""
Paper figures at IEEE width (column 3.5 in, page 7.16 in), as vector PDF + 300 dpi PNG.
Fonts are embedded as TrueType (pdf.fonttype 42), which IEEE PDF checks require.

In a Kaggle cell:
    os.environ["OUT_DIR"] = "<folder holding probs_*.csv>"
    %run make_figures.py
Writes to FIG_DIR (default /kaggle/working/figs). Download that folder.

fig1_per_fst        sensitivity and AUC by Fitzpatrick level, mean +/- SD, dots = seeds
fig2_subtype        sensitivity by cancer subtype and skin group (needs the CSV; descriptive)
fig3_recovery       FST 6 recovery: baseline vs +FST 4,5 (holdout) vs oversample FST 6,
                    each compared on exactly the same test images
"""
import glob
import os
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

OUT  = os.environ.get("OUT_DIR", "/kaggle/working")
DATA = os.environ.get("DATA_ROOT", "/kaggle/input")
FIG  = os.environ.get("FIG_DIR", "/kaggle/working/figs")
os.makedirs(FIG, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "font.size": 7.5, "axes.labelsize": 7.5, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 6.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 300,
})
BASE, HI, REF = "#7A9CC6", "#D55E00", "#333333"      # FST 6 gets colour AND hatching
GRP_COL = {"FST1-3": "#7A9CC6", "FST4-5": "#E69F00", "FST6": "#D55E00"}
ROMAN = {1: "I", 2: "II", 3: "III", 4: "IV", 5: "V", 6: "VI"}


def sens(p, y):
    m = y == 1
    return float((p[m] >= 0.5).mean()) if m.any() else np.nan


def auc(p, y):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan


def read(suffix):
    found = {}
    for f in glob.glob(f"{OUT}/probs_seed*{suffix}.csv"):
        m = re.fullmatch(rf"probs_seed(\d+){re.escape(suffix)}\.csv", os.path.basename(f))
        if m:
            found[int(m.group(1))] = pd.read_csv(f)
    return dict(sorted(found.items()))


main, mit, hold = read(""), read("_mitigate6"), read("_hold45")
hold_sm = read("_hold45_sm")
if not main:
    raise SystemExit(f"No probs_seed*.csv in {OUT}")


def save(fig, name):
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.png", bbox_inches="tight")
    plt.close(fig)
    print("wrote", f"{FIG}/{name}.pdf / .png")


def dots(ax, x, vals, color="black"):
    vals = np.asarray(vals, float)
    off = np.linspace(-0.14, 0.14, len(vals)) if len(vals) > 1 else np.zeros(1)
    ax.scatter(x + off, vals, s=5, color=color, zorder=3, linewidths=0)


# ---------------------------------------------------------------- Figure 1
rows = []
for s, d in main.items():
    for lvl, g in d.groupby("fst"):
        p, y = g.prob.values, g.y.values
        rows.append(dict(seed=s, fst=int(lvl), sens=sens(p, y), auc=auc(p, y)))
    L = d[d.fst <= 3]
    rows.append(dict(seed=s, fst=0, sens=sens(L.prob.values, L.y.values),
                     auc=auc(L.prob.values, L.y.values)))      # fst 0 = pooled FST 1-3
T = pd.DataFrame(rows)
levels = [l for l in sorted(T.fst.unique()) if l > 0]

fig, axes = plt.subplots(1, 2, figsize=(7.16, 2.3))
for ax, col, ylab in ((axes[0], "sens", "Sensitivity"), (axes[1], "auc", "AUC")):
    for i, l in enumerate(levels):
        v = T[T.fst == l][col].values
        ax.bar(i, v.mean(), yerr=v.std(ddof=1) if len(v) > 1 else 0, capsize=2,
               color=HI if l == 6 else BASE, hatch="///" if l == 6 else None,
               edgecolor="black", linewidth=0.5, error_kw=dict(lw=0.6))
        dots(ax, i, v)
    ref = T[T.fst == 0][col].mean()
    ax.axhline(ref, ls="--", lw=0.7, color=REF)
    ax.text(len(levels) - 0.5, ref + 0.02, "FST I–III pooled", ha="right", va="bottom", fontsize=6.5)
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels([ROMAN[l] for l in levels])
    ax.set_xlabel("Fitzpatrick skin type")
    ax.set_ylabel(ylab)
    ax.set_ylim(0, 1)
axes[0].set_title("(a) Sensitivity at threshold 0.5", loc="left")
axes[1].set_title("(b) AUC", loc="left")
fig.tight_layout()
save(fig, "fig1_per_fst")

# ---------------------------------------------------------------- Figure 2
csvs = [c for c in glob.glob(f"{DATA}/**/*.csv", recursive=True)
        if "fitz" in os.path.basename(c).lower()]
if os.environ.get("CSV_PATH"):
    csvs = [os.environ["CSV_PATH"]]
if csvs:
    meta = (pd.read_csv(csvs[0])[["md5hash", "label"]]
            .rename(columns={"label": "dx"}).drop_duplicates("md5hash"))

    def family(dx):
        s = str(dx).lower()
        if "kaposi" in s:
            return "Kaposi\nsarcoma"
        if "squamous" in s:
            return "Squamous\ncell"
        if "basal cell" in s:
            return "Basal\ncell"
        if "melanoma" in s or "lentigo maligna" in s:
            return "Melanoma"
        return None

    per = []
    for s, d in main.items():
        d = d.copy()
        d["md5hash"] = d["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])
        d = d.merge(meta, on="md5hash", how="left")
        d = d[d.y == 1]
        d["fam"] = d["dx"].map(family)
        d = d.dropna(subset=["fam"])
        d["grp"] = np.where(d.fst <= 3, "FST1-3", np.where(d.fst <= 5, "FST4-5", "FST6"))
        d["hit"] = d.prob >= 0.5
        g = d.groupby(["fam", "grp"]).agg(n=("hit", "size"), sens=("hit", "mean")).reset_index()
        g["seed"] = s
        per.append(g)
    R = pd.concat(per)
    S = R.pivot_table(index="fam", columns="grp", values="sens", aggfunc="mean")
    N = R.pivot_table(index="fam", columns="grp", values="n", aggfunc="mean")
    order = [f for f in N.sort_values("FST6", ascending=False).index if N.loc[f, "FST6"] >= 5]
    fig, ax = plt.subplots(figsize=(3.5, 2.3))
    w = 0.26
    for j, grp in enumerate(["FST1-3", "FST4-5", "FST6"]):
        xs = np.arange(len(order)) + (j - 1) * w
        ax.bar(xs, [S.loc[f, grp] for f in order], w, color=GRP_COL[grp],
               hatch="///" if grp == "FST6" else None, edgecolor="black", linewidth=0.5,
               label={"FST1-3": "FST I–III", "FST4-5": "FST IV–V", "FST6": "FST VI"}[grp])
    for i, f in enumerate(order):
        ax.text(i + w, S.loc[f, "FST6"] + 0.02, f"n={int(round(N.loc[f, 'FST6']))}",
                ha="center", va="bottom", fontsize=5.5)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, fontsize=6.5)
    ax.set_ylabel("Sensitivity")
    ax.set_ylim(0, 1)
    ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))
    fig.tight_layout()
    save(fig, "fig2_subtype")
else:
    print("Fitzpatrick17k CSV not found - skipped fig2_subtype")

# ---------------------------------------------------------------- Figure 3
pairs = []     # (label, [per-seed sens], [per-seed auc], colour, hatch)
if hold or mit:
    def collect(src, subset_fn):
        out = []
        for s, m in src.items():
            if s not in main:
                continue
            b, t = subset_fn(main[s], m)
            out.append((sens(b.prob.values, b.y.values), auc(b.prob.values, b.y.values),
                        sens(t.prob.values, t.y.values), auc(t.prob.values, t.y.values)))
        return np.array(out)

    def hold_fn(b, h):
        return b[b.fst == 6], h[h.fst == 6]

    def mit_fn(b, m):
        m6 = m[m.fst == 6]
        return b[(b.fst == 6) & b.path.isin(m6.path)], m6[m6.path.isin(b.path)]

    xs, labels, sv, av, cols, centres = [], [], [], [], [], []
    x = 0.0
    groups = (("all FST VI images", hold_fn,
               (("+ IV–V\nmore data", hold), ("+ IV–V\nsame size", hold_sm))),
              ("held-out half of FST VI", mit_fn, (("oversample\nVI", mit),)))
    for name, fn, treats in groups:
        arms = [(t, collect(src, fn)) for t, src in treats if src]
        arms = [(t, r) for t, r in arms if len(r)]
        if not arms:
            continue
        # baseline uses the seeds of the first arm; every arm is tested on the same images
        r0 = arms[0][1]
        bars = [("baseline", r0[:, 0], r0[:, 1], BASE)]
        for t, r in arms:
            bars.append((t, r[:, 2], r[:, 3], HI))
        for k, (lab, sv_, av_, c) in enumerate(bars):
            xs.append(x + k); labels.append(lab); sv.append(sv_); av.append(av_); cols.append(c)
        centres.append((x + (len(bars) - 1) / 2, "tested on\n" + name))
        x += len(bars) + 0.8
    ref_s = np.mean([sens(d[d.fst <= 3].prob.values, d[d.fst <= 3].y.values) for d in main.values()])
    ref_a = np.mean([auc(d[d.fst <= 3].prob.values, d[d.fst <= 3].y.values) for d in main.values()])
    fig, axes = plt.subplots(1, 2, figsize=(7.16, 2.4))
    for ax, vals, ref, ylab in ((axes[0], sv, ref_s, "FST VI sensitivity"), (axes[1], av, ref_a, "FST VI AUC")):
        for xi, v, c in zip(xs, vals, cols):
            ax.bar(xi, v.mean(), 0.8, yerr=v.std(ddof=1) if len(v) > 1 else 0, capsize=2, color=c,
                   hatch="///" if c == HI else None, edgecolor="black", linewidth=0.5,
                   error_kw=dict(lw=0.6))
            dots(ax, xi, v)
        ax.axhline(ref, ls="--", lw=0.7, color=REF)
        ax.text(xs[-1] + 0.5, ref + 0.02, "FST I–III", ha="right", va="bottom", fontsize=6.5)
        ax.set_xticks(xs)
        ax.set_xticklabels(labels, fontsize=6)
        for cx, cap in centres:
            ax.annotate(cap, xy=(cx, 0), xycoords=("data", "axes fraction"),
                        xytext=(0, -30), textcoords="offset points",
                        ha="center", va="top", fontsize=6.5, style="italic", linespacing=1.15)
        ax.set_ylabel(ylab)
        ax.set_ylim(0, 1)
    axes[0].set_title("(a) Sensitivity", loc="left")
    axes[1].set_title("(b) AUC", loc="left")
    fig.tight_layout()
    save(fig, "fig3_recovery")
else:
    print("no holdout/mitigation files - skipped fig3_recovery")

## 3. Duplicate check (Table II: 'duplicates removed' and 'Fitzpatrick17k-C' rows)

In [ ]:
!python analyze_dups.py zipin

In [ ]:
!python analyze_dups.py zipin _effb0

## 4. Interventions on leak-free FST VI images (Table III) and Fig. 6

In [ ]:
!python intervention_leakfree.py

In [ ]:
import os, shutil, subprocess
src = open("make_figures.py").read().replace("all FST VI images", "530 leak-free FST VI images")
open("make_figures_lf.py", "w").write(src)
env = dict(os.environ, OUT_DIR="/kaggle/working/lf/out", DATA_ROOT="/kaggle/working",
           CSV_PATH="/kaggle/working/official.csv", FIG_DIR="/kaggle/working/lf/figs")
print(subprocess.run(["python", "make_figures_lf.py"], env=env, capture_output=True, text=True).stdout[-300:])
os.makedirs("figs_final", exist_ok=True)
for ext in ("pdf", "png"):
    shutil.copy(f"lf/figs/fig3_recovery.{ext}", f"figs_final/fig_interventions_leakfree.{ext}")

## 5. DDI at matched specificity (Table IV, last column)

In [ ]:
!python ddi_matched.py probs_ddi_all.csv

## 6. Study design, grouping comparison and robustness forest plot

In [ ]:
!python new_figs.py
import glob, shutil
for f in glob.glob("figs_extra/*"):
    shutil.copy(f, "figs_final/")
print(sorted(os.listdir("figs_final")))

## Output
`figs_final/` holds the new figures (PDF for the paper, PNG to preview):
`fig_design`, `fig_grouping`, `fig_forest`, `fig_interventions_leakfree`.
Per-type and subtype figures come from notebook 02 (`figs/fig1_per_fst`, `figs/fig2_subtype`).

## 7. Pooled statistics over seeds
Main per-type estimates with Wilson/bootstrap intervals, contrasts, matched specificity, trend test, site regression, subtype intervals, negative-class reweighting and paired intervention differences. Needs `statsmodels` (preinstalled on Kaggle).

In [ ]:
%%writefile posthoc2.py
"""Reviewer-response analyses on existing light-only predictions (no training).
Pooled estimates with image-level CIs, site x FST, site-adjusted regression,
matched specificity, trend test, subtype CIs, intervention differences, negative mix.
Run in a folder holding zipin/probs_seed*.csv, official.csv, complete_clusters.txt, f17k_names.csv."""
import ast, glob, os, re
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
import statsmodels.formula.api as smf

NB = int(os.environ.get("NBOOT", 2000)); rng = np.random.default_rng(0)
pd.set_option("display.width", 220, "display.precision", 3)
stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
off = pd.read_csv("official.csv"); off = off[off.fitzpatrick_scale.between(1, 6)].copy()
off["site"] = off.url.astype(str).str.extract(r"https?://(?:www\.)?([^/]+)")[0]
off["site"] = np.where(off.site.isin(["dermaamin.com", "atlasdermatologico.com.br"]), off.site, "other")
lab = off.label.str.lower()
off["fam"] = np.select([lab.str.contains("melanoma") | lab.str.contains("lentigo maligna"),
                        lab.str.contains("squamous"), lab.str.contains("basal"),
                        lab.str.contains("kaposi"), lab.str.contains("mycosis")],
                       ["melanoma", "squamous cell carcinoma", "basal cell carcinoma",
                        "kaposi sarcoma", "mycosis fungoides"], "other")
meta = off.set_index("md5hash")[["site", "fam", "nine_partition_label"]]


def wilson(k, n, z=1.959964):
    if n == 0: return np.nan, np.nan
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def load(suf):
    R = []
    for f in glob.glob(f"zipin/probs_seed*{suf}.csv"):
        m = re.fullmatch(rf"probs_seed(\d+){re.escape(suf)}\.csv", os.path.basename(f))
        if m:
            d = pd.read_csv(f); d["h"] = d.path.map(stem); d["seed"] = int(m.group(1)); R.append(d)
    A = pd.concat(R)
    # pool: average each image's score over the seeds in which it was tested
    P = A.groupby("h").agg(prob=("prob", "mean"), y=("y", "first"), fst=("fst", "first"),
                           nseeds=("seed", "nunique")).reset_index()
    return P.join(meta, on="h")


sens = lambda p, y: float((p[y == 1] >= .5).mean())
auc = lambda p, y: float(roc_auc_score(y, p))
grp = lambda f: np.where(f <= 3, "I-III", np.where(f <= 5, "IV-V", "VI"))


def boot_ci(fn, d, nb=NB):
    v = [fn(d.iloc[rng.integers(0, len(d), len(d))]) for _ in range(nb)]
    return np.nanpercentile(v, [2.5, 97.5])


for suf, name in (("", "RESNET-50"), ("_effb0", "EFFICIENTNET-B0")):
    P = load(suf); P["g"] = grp(P.fst)
    print("=" * 90 + f"\n{name}: POOLED OVER SEEDS (mean score per image), image-level 95% CIs\n" + "=" * 90)
    rows = []
    for f, d in P.groupby("fst"):
        k, n = int((d.prob[d.y == 1] >= .5).sum()), int(d.y.sum()); lo, hi = wilson(k, n)
        al, ah = boot_ci(lambda x: auc(x.prob.values, x.y.values), d)
        rows.append(dict(fst=int(f), images=len(d), malignant=n, sens=k / n, sens_ci=f"[{lo:.2f}, {hi:.2f}]",
                         spec=float((d.prob[d.y == 0] < .5).mean()), auc=auc(d.prob.values, d.y.values),
                         auc_ci=f"[{al:.2f}, {ah:.2f}]"))
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    for lo_, hi_ in (("I-III", "VI"), ("I-III", "IV-V"), ("IV-V", "VI")):
        A, B = P[P.g == lo_], P[P.g == hi_]
        for mname, st in (("sensitivity", sens), ("AUC", auc)):
            pt = st(A.prob.values, A.y.values) - st(B.prob.values, B.y.values)
            bs = [st(A.prob.values[i], A.y.values[i]) - st(B.prob.values[j], B.y.values[j])
                  for i, j in ((rng.integers(0, len(A), len(A)), rng.integers(0, len(B), len(B))) for _ in range(NB))]
            l, h = np.percentile(bs, [2.5, 97.5])
            print(f"  {lo_:>5s} minus {hi_:<4s} {mname:11s} {pt:+.3f}  95% CI [{l:+.3f}, {h:+.3f}]")
    if suf: continue

    # ---------------------------------------------------------------- matched specificity (common threshold)
    print("\nMATCHED SPECIFICITY: one threshold set on FST I-III negatives, applied to every type")
    L = P[(P.g == "I-III") & (P.y == 0)].prob.values
    for sp in (0.80, 0.90):
        t = np.quantile(L, sp)
        out = {int(f): f"{(d.prob[d.y == 1] > t).mean():.3f} (spec {(d.prob[d.y == 0] <= t).mean():.2f})"
               for f, d in P.groupby("fst")}
        print(f"  spec {sp:.0%} on I-III: sensitivity by FST {out}")
    print("  within-type matched specificity (threshold set inside each type):")
    for sp in (0.80, 0.90):
        out = {}
        for f, d in P.groupby("fst"):
            t = np.quantile(d.prob[d.y == 0].values, sp); out[int(f)] = round(float((d.prob[d.y == 1] > t).mean()), 3)
        print(f"    spec {sp:.0%}: {out}")

    # ---------------------------------------------------------------- trend test
    print("\nTREND TEST: linear fit of sensitivity on FST I-V, extrapolated to VI (bootstrap over images)")
    def trend(d):
        s = [sens(x.prob.values, x.y.values) for _, x in d[d.fst <= 5].groupby("fst")]
        b = np.polyfit(np.arange(1, 6), s, 1); pred6 = np.polyval(b, 6)
        return pred6, sens(d[d.fst == 6].prob.values, d[d.fst == 6].y.values)
    p6, o6 = trend(P)
    bs = []
    for _ in range(NB):
        d = pd.concat([x.iloc[rng.integers(0, len(x), len(x))] for _, x in P.groupby("fst")])
        a, b = trend(d); bs.append(b - a)
    l, h = np.percentile(bs, [2.5, 97.5])
    print(f"  predicted FST VI {p6:.3f}, observed {o6:.3f}, observed minus predicted {o6 - p6:+.3f} [{l:+.3f}, {h:+.3f}]")

    # ---------------------------------------------------------------- site x FST
    print("\nSITE x SKIN GROUP (pooled): malignant n, sensitivity [Wilson CI], AUC")
    rows = []
    for (s, g), d in P[P.site != "other"].groupby(["site", "g"]):
        k, n = int((d.prob[d.y == 1] >= .5).sum()), int(d.y.sum()); lo, hi = wilson(k, n)
        rows.append(dict(site=s, group=g, images=len(d), malignant=n, sens=round(k / n, 3),
                         sens_ci=f"[{lo:.2f}, {hi:.2f}]", auc=round(auc(d.prob.values, d.y.values), 3)))
    print(pd.DataFrame(rows).to_string(index=False))
    M = P[(P.y == 1) & (P.site != "other")].copy()
    M["det"] = (M.prob >= .5).astype(int)
    M["g"] = pd.Categorical(M.g, ["I-III", "IV-V", "VI"])
    for form in ("det ~ C(g)", "det ~ C(g) + C(site)", "det ~ C(g) + C(site) + C(fam)"):
        r = smf.logit(form, M).fit(disp=0)
        orr = np.exp(r.params["C(g)[T.VI]"]); ci = np.exp(r.conf_int().loc["C(g)[T.VI]"])
        print(f"  {form:32s} odds ratio of detection, FST VI vs I-III: {orr:.2f} [{ci[0]:.2f}, {ci[1]:.2f}]  p={r.pvalues['C(g)[T.VI]']:.4f}")
    D = P[P.site == "dermaamin.com"]
    k6 = D[(D.fst == 6) & (D.y == 1)]
    print(f"  within dermaamin.com: FST VI sens {sens(k6.prob.values, k6.y.values):.3f} (n={len(k6)}) "
          f"vs FST I-III {sens(D[D.g == 'I-III'].prob.values, D[D.g == 'I-III'].y.values):.3f}")

    # ---------------------------------------------------------------- subtype CIs
    print("\nSUBTYPE x GROUP sensitivity [Wilson CI] (malignant n)")
    rows = []
    for (fam, g), d in P[P.y == 1].groupby(["fam", "g"]):
        k, n = int((d.prob >= .5).sum()), len(d); lo, hi = wilson(k, n)
        rows.append(dict(subtype=fam, group=g, n=n, sens=f"{k / n:.2f} [{lo:.2f}, {hi:.2f}]"))
    print(pd.DataFrame(rows).pivot(index="subtype", columns="group", values="sens").to_string())
    print(pd.DataFrame(rows).pivot(index="subtype", columns="group", values="n").to_string())

    # ---------------------------------------------------------------- negative class mix
    print("\nNON-MALIGNANT CLASS MIX by group (nine-partition label, % of negatives)")
    Ng = P[P.y == 0]
    print((pd.crosstab(Ng.nine_partition_label, Ng.g, normalize="columns") * 100).round(1).to_string())
    print("  malignant prevalence:", {g: round(d.y.mean(), 3) for g, d in P.groupby("g")})
    ref = Ng[Ng.g == "I-III"].nine_partition_label.value_counts(normalize=True)
    def std_auc(d):
        neg = d[d.y == 0]; w_neg = neg.nine_partition_label.map(ref / neg.nine_partition_label.value_counts(normalize=True)).fillna(0)
        w = pd.concat([pd.Series(1.0, index=d[d.y == 1].index), w_neg])
        x = d.loc[w.index]
        return roc_auc_score(x.y, x.prob, sample_weight=w.values)
    for g in ("IV-V", "VI"):
        d = P[P.g == g]
        print(f"  FST {g}: AUC {auc(d.prob.values, d.y.values):.3f}; with negatives reweighted to the I-III mix {std_auc(d):.3f}")

# ---------------------------------------------------------------- interventions: paired differences
print("\n" + "=" * 90 + "\nINTERVENTIONS: paired differences on the same leak-free FST VI images (seed-averaged scores)\n" + "=" * 90)
n2m = dict(zip(*[pd.read_csv("f17k_names.csv")[c] for c in ("filename", "md5hash")]))
cl = [[n2m[n] for n in ast.literal_eval(l) if n in n2m] for l in open("complete_clusters.txt") if l.strip()]
cid = {h: i for i, c in enumerate(cl) for h in c}; mem = {i: set(c) for i, c in enumerate(cl)}
ivv = set(off.md5hash[off.fitzpatrick_scale.isin([4, 5])])
leak = {h for h in off.md5hash[off.fitzpatrick_scale == 6] if (i := cid.get(h)) is not None and (mem[i] - {h}) & ivv}
S = {}
for suf in ("", "_hold45", "_hold45_sm"):
    R = pd.concat([pd.read_csv(f"zipin/probs_seed{s}{suf}.csv").assign(seed=s) for s in range(3)])
    R = R[R.fst == 6]; R["h"] = R.path.map(stem)
    S[suf] = R.groupby("h").agg(prob=("prob", "mean"), y=("y", "first"))
idx = sorted(set(S[""].index) - leak)
base = S[""].loc[idx]
print(f"leak-free FST VI images: {len(idx)} ({int(base.y.sum())} malignant)")
for suf, nm in (("_hold45", "+FST IV-V"), ("_hold45_sm", "+FST IV-V size-matched")):
    X = S[suf].loc[idx]
    for mname, st in (("sensitivity", sens), ("AUC", auc)):
        pt = st(X.prob.values, X.y.values) - st(base.prob.values, base.y.values)
        bs = []
        for _ in range(NB):
            i = rng.integers(0, len(idx), len(idx))
            bs.append(st(X.prob.values[i], X.y.values[i]) - st(base.prob.values[i], base.y.values[i]))
        l, h = np.percentile(bs, [2.5, 97.5])
        print(f"  {nm:24s} minus baseline  {mname:11s} {pt:+.3f}  95% CI [{l:+.3f}, {h:+.3f}]")
Xa, Xb = S["_hold45"].loc[idx], S["_hold45_sm"].loc[idx]
for mname, st in (("sensitivity", sens), ("AUC", auc)):
    pt = st(Xa.prob.values, Xa.y.values) - st(Xb.prob.values, Xb.y.values)
    bs = [st(Xa.prob.values[i], Xa.y.values[i]) - st(Xb.prob.values[i], Xb.y.values[i])
          for i in (rng.integers(0, len(idx), len(idx)) for _ in range(NB))]
    l, h = np.percentile(bs, [2.5, 97.5])
    print(f"  full minus size-matched             {mname:11s} {pt:+.3f}  95% CI [{l:+.3f}, {h:+.3f}]")


In [ ]:
!python posthoc2.py | tee posthoc2.log

## 8. Per-type figure (light-only vs all-skin-type training)
Plots the values printed in section 7 and by `analyze_cv.py` in notebook 05.

In [ ]:
%%writefile fig_per_fst2.py
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np
plt.rcParams.update({"font.size":7.5,"font.family":"serif","axes.spines.top":False,"axes.spines.right":False})
fst=["I","II","III","IV","V","VI"]
# light-only ResNet-50, seed-averaged scores (posthoc2.log)
L_s=[.732,.752,.777,.694,.592,.410]; L_sl=[.68,.71,.73,.64,.51,.30]; L_sh=[.78,.79,.82,.74,.67,.54]
L_a=[.925,.919,.936,.946,.901,.810]; L_al=[.91,.90,.92,.93,.87,.75]; L_ah=[.94,.93,.95,.96,.93,.87]
# all-skin-type 5-fold CV (notebook 05)
C_s=[.726,.739,.768,.774,.653,.475]; C_sl=[.683,.706,.727,.724,.573,.355]; C_sh=[.765,.769,.804,.818,.725,.598]
C_a=[.897,.916,.932,.947,.921,.848]; C_al=[.880,.905,.916,.933,.894,.787]; C_ah=[.914,.927,.945,.960,.944,.901]
x=np.arange(6); w=.36
fig,ax=plt.subplots(1,2,figsize=(3.45,1.9))
for a,(Lv,Ll,Lh,Cv,Cl,Ch,t,yl) in zip(ax,[(L_s,L_sl,L_sh,C_s,C_sl,C_sh,"(a) Sensitivity",(0,1)),(L_a,L_al,L_ah,C_a,C_al,C_ah,"(b) AUC",(0.5,1))]):
    for off,v,lo,hi,c,lab,h in [(-w/2,Lv,Ll,Lh,"#4a6fa5","Trained on FST I–III",None),(w/2,Cv,Cl,Ch,"#d08c3c","Trained on all types (CV)",None)]:
        v,lo,hi=map(np.array,(v,lo,hi))
        b=a.bar(x+off,v-yl[0],w,bottom=yl[0],color=c,label=lab,edgecolor="none")
        a.errorbar(x+off,v,yerr=[v-lo,hi-v],fmt="none",ecolor="k",elinewidth=.6,capsize=1.2)
    a.set_xticks(x); a.set_xticklabels(fst); a.set_ylim(*yl); a.set_title(t,fontsize=7.5,loc="left")
    a.set_xlabel("Fitzpatrick type"); a.axvspan(4.5,5.5,color="0.9",zorder=-1)
h,l=ax[0].get_legend_handles_labels(); fig.legend(h,l,frameon=False,fontsize=6.5,loc="upper center",ncol=2,handlelength=1,bbox_to_anchor=(0.5,1.0))
fig.tight_layout(pad=.3,w_pad=.8,rect=(0,0,1,.9))
fig.savefig("figs_final/fig_per_fst2.pdf"); fig.savefig("figs_final/fig_per_fst2.png",dpi=200)


In [ ]:
import os; os.makedirs('figs_final', exist_ok=True)
!python fig_per_fst2.py